In [ ]:
from dataclasses import dataclass
import math
import random
import collections
import numpy as np
import os
from pathlib import Path
from PIL import Image
from PIL.ExifTags import TAGS
from collections import Counter
import tqdm
import re
import torch
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, fcluster
import time
from datetime import datetime
import exiftool
import enum
import torchvision

raise NotImplementedError('use eda00.py')


In [2]:
DATA_ROOT = '/home/slavik/e202602_eclipse/data'
BRIGHTNESS_MIN = 0.0
BRIGHTNESS_MAX = 1.0
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [3]:
class MoonInfoOrigin(enum.Enum):
    DIRECT = 0
    INTERPOLATED = 1

In [4]:

@dataclass
class ImageInfo:
    path: Path
    width: int
    height: int
    avg_brightness: float
    timestamp: float
    exposure_time: float
    moon: tuple[float, float, float] = None    # (center_i, center_j, radius) in pixels
    moon_info_origin: MoonInfoOrigin = None
    moon_pos_std_px: float = None



In [5]:
def get_info_from_exif(img_path: Path) -> float:
    """
    Extract exposure time in seconds from image EXIF via exiftool.
    Raises ValueError if EXIF:ExposureTime is missing.
    """
    with exiftool.ExifToolHelper() as et:
        metadata = et.get_metadata(str(img_path))[0]
    exposure_time = metadata.get('EXIF:ExposureTime')
    assert exposure_time is not None
    assert isinstance(exposure_time, (int, float)), type(exposure_time)
    # Composite:SubSecDateTimeOriginal 2024:04:08 15:27:05.89-04:00
    subsec_date_time_original = metadata.get('Composite:SubSecDateTimeOriginal')
    assert subsec_date_time_original is not None
    assert re.match(r'\d{4}:\d{2}:\d{2} \d{2}:\d{2}:\d{2}\.\d{2}-\d{2}:\d{2}', subsec_date_time_original), subsec_date_time_original
    expected_format = "%Y:%m:%d %H:%M:%S.%f%z"
    try:
        dt = datetime.strptime(subsec_date_time_original, expected_format)
        timestamp = dt.timestamp()
    except ValueError as e:
        raise ValueError(f"Failed to parse DateTime '{subsec_date_time_original}' in {img_path}: {e}")
    return float(exposure_time), timestamp


def get_image_infos():
    jpg_files = list(Path(DATA_ROOT).rglob('*.jpg')) + list(Path(DATA_ROOT).rglob('*.JPG'))
    image_infos = []
    for jpg_file in tqdm.tqdm(jpg_files, desc="First scan of images"):
        with Image.open(jpg_file) as img:
            width, height = img.size
            avg_brightness = np.array(img).astype(np.float32).mean() / 255.0
            if BRIGHTNESS_MIN <= avg_brightness <= BRIGHTNESS_MAX:
                exposure_time, timestamp = get_info_from_exif(jpg_file)
                image_infos.append(ImageInfo(path=jpg_file, width=width, height=height, avg_brightness=avg_brightness, timestamp=timestamp, exposure_time=exposure_time))
    assert len(image_infos) > 0
    for ii in image_infos:
        assert ii.width == image_infos[0].width
        assert ii.height == image_infos[0].height
    image_infos.sort(key=lambda x: x.avg_brightness)
    return image_infos

In [6]:
N_SECTORS = 360
N_TRIPLETS = 1024
N_CLUSTER = 256
DEBUG_RADIUS_PX = 4
REFINE_ITERATIONS = 3
MIN_TRIPLET_DEGREES = 30


def _indices_within_degrees(center: int, deg: int) -> set:
    """Sector indices within deg degrees of center (wrap-around)."""
    return {(center + d) % N_SECTORS for d in range(-(deg - 1), deg)}


def sample_triplet_indices(n_pts: int, min_degrees: int = MIN_TRIPLET_DEGREES) -> tuple[int, int, int]:
    """
    Sample three distinct sector indices such that each pair is at least min_degrees apart.
    Falls back to unrestricted random triplet if not enough spread is available.
    """
    available = set(range(n_pts))
    a = random.sample(list(available), 1)[0]
    available -= _indices_within_degrees(a, min_degrees) & available
    assert len(available) >= 2
    b = random.sample(list(available), 1)[0]
    available -= _indices_within_degrees(b, min_degrees) & available
    assert len(available) >= 1
    c = random.sample(list(available), 1)[0]
    return (a, b, c)


def refine_moon(img: torch.Tensor, center_i: float, center_j: float) -> tuple[float, float, float]:
    """
    Refine moon center from image and current center (i, j).
    img: (H, W, 3) float32 [0,1] on GPU.
    Returns (i, j, radius) as tuple of (float, float, float).
    """
    assert img.ndim == 3 and img.shape[2] == 3
    H, W = img.shape[0], img.shape[1]
    dev = img.device
    img_size = float(max(H, W))

    gray = img.mean(dim=2)
    sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32, device=dev).view(1, 1, 3, 3)
    sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32, device=dev).view(1, 1, 3, 3)
    g = gray.unsqueeze(0).unsqueeze(0)
    grad_x = torch.nn.functional.conv2d(g, sobel_x, padding=1).squeeze()
    grad_y = torch.nn.functional.conv2d(g, sobel_y, padding=1).squeeze()

    dy = torch.arange(H, device=dev, dtype=torch.float32).view(-1, 1) - center_i
    dx = torch.arange(W, device=dev, dtype=torch.float32).view(1, -1) - center_j
    norm = torch.sqrt(dx * dx + dy * dy).clamp(min=1e-6)
    u_x = dx / norm
    u_y = dy / norm

    angle = torch.atan2(dy, dx)
    sector_id = (torch.floor((angle + math.pi) / (2 * math.pi) * N_SECTORS).long() % N_SECTORS)

    dot_product = grad_x * u_x + grad_y * u_y
    dot_product_flat = dot_product.reshape(-1)
    sector_flat = sector_id.reshape(-1)
    W_t = W

    points_list = []
    for s in range(N_SECTORS):
        mask = sector_flat == s
        if mask.any():
            masked = torch.where(mask, dot_product_flat, torch.tensor(-1e9, device=dev, dtype=torch.float32))
            idx = masked.argmax().item()
            i, j = idx // W_t, idx % W_t
            points_list.append((i, j))

    n_pts = len(points_list)
    if n_pts < 3:
        return (center_i, center_j, 0.0)

    def circumcenter(i1, j1, i2, j2, i3, j3):
        x1, y1, x2, y2, x3, y3 = float(j1), float(i1), float(j2), float(i2), float(j3), float(i3)
        D = 2.0 * (x1 * (y2 - y3) + x2 * (y3 - y1) + x3 * (y1 - y2))
        if abs(D) < 1e-10:
            return None
        ox = ((x1 * x1 + y1 * y1) * (y2 - y3) + (x2 * x2 + y2 * y2) * (y3 - y1) + (x3 * x3 + y3 * y3) * (y1 - y2)) / D
        oy = ((x1 * x1 + y1 * y1) * (x3 - x2) + (x2 * x2 + y2 * y2) * (x1 - x3) + (x3 * x3 + y3 * y3) * (x2 - x1)) / D
        oi, oj = oy, ox
        d1 = math.hypot(i1 - oi, j1 - oj)
        d2 = math.hypot(i2 - oi, j2 - oj)
        d3 = math.hypot(i3 - oi, j3 - oj)
        if d1 > img_size or d2 > img_size or d3 > img_size:
            return None
        # Compute radius as average of distances from circumcenter to the 3 points
        radius = (d1 + d2 + d3) / 3.0
        return (oi, oj, radius)

    circumcenters = []
    radii = []
    while len(circumcenters) < N_TRIPLETS:
        a, b, c = sample_triplet_indices(n_pts)
        i1, j1 = points_list[a]
        i2, j2 = points_list[b]
        i3, j3 = points_list[c]
        cc_result = circumcenter(i1, j1, i2, j2, i3, j3)
        if cc_result is not None:
            oi, oj, radius = cc_result
            circumcenters.append((oi, oj))
            radii.append(radius)

    pts = np.array(circumcenters, dtype=np.float64)
    Z = linkage(pts, method="complete")
    t_lo, t_hi = 0.0, float(Z[-1, 2])
    for _ in range(60):
        t = (t_lo + t_hi) / 2
        labels = fcluster(Z, t, criterion="distance")
        sizes = np.bincount(labels)
        max_size = int(sizes.max())
        if max_size >= N_CLUSTER:
            t_hi = t
        else:
            t_lo = t
    labels = fcluster(Z, t_hi, criterion="distance")
    sizes = np.bincount(labels)
    which = int(np.argmax(sizes))
    cluster_mask = labels == which
    cluster_pts = pts[cluster_mask]
    cluster_radii = np.array(radii)[cluster_mask]
    ci = float(cluster_pts[:, 0].mean())
    cj = float(cluster_pts[:, 1].mean())
    radius = float(cluster_radii.mean())
    return (ci, cj, radius)


def find_moon(img: torch.Tensor, i0: float, j0: float) -> tuple[float, float, float]:
    """
    Find moon center by iteratively refining from image center.
    img: (H, W, 3) float32 [0,1] on GPU. Returns (i, j, radius) as tuple of (float, float, float).
    """
    assert img.ndim == 3 and img.shape[2] == 3
    center_i, center_j = i0, j0
    radius = 0.0
    for _ in range(REFINE_ITERATIONS):
        center_i, center_j, radius = refine_moon(img, center_i, center_j)

    if False:
        img_np = img.cpu().numpy()
        H, W = img_np.shape[0], img_np.shape[1]
        
        # Calculate crop size: 2.2 * radius (10% padding on each side)
        half_crop = int(round(1.1 * radius))
        
        # Calculate crop bounds (square crop centered on moon)
        i_min = int(round(center_i - half_crop))
        i_max = int(round(center_i + half_crop))
        j_min = int(round(center_j - half_crop))
        j_max = int(round(center_j + half_crop))
        
        # Check bounds and throw exception if out of bounds
        if i_min < 0 or i_max >= H or j_min < 0 or j_max >= W:
            raise ValueError(f"Crop bounds out of image: half_crop={half_crop}, center=({center_i}, {center_j}), radius={radius:.1f}, image_size=({H}, {W}), bounds=({i_min}, {i_max}, {j_min}, {j_max})")
        
        # Crop image
        img_cropped = img_np[i_min:i_max+1, j_min:j_max+1]
        
        # Create figure
        plt.figure(figsize=(12, 12))
        plt.imshow(img_cropped)
        
        # Draw center as small green circle (relative to cropped image)
        center_j_crop = center_j - j_min
        center_i_crop = center_i - i_min
        plt.gca().add_patch(plt.Circle((center_j_crop, center_i_crop), DEBUG_RADIUS_PX, color="green", fill=True))
        
        # Draw 36 equally spaced green pixels on the circle border
        n_points = 36
        for k in range(n_points):
            angle = 2 * math.pi * k / n_points
            border_j = center_j_crop + radius * math.cos(angle)
            border_i = center_i_crop + radius * math.sin(angle)
            border_j_int = int(round(border_j))
            border_i_int = int(round(border_i))
            # Draw single green pixel
            plt.plot(border_j_int, border_i_int, 'g.', markersize=1)
        
        plt.title(f"Moon center: (i={center_i}, j={center_j}), radius: {radius:.1f}")
        plt.axis("off")
        plt.tight_layout()
        plt.show()
    return (center_i, center_j, radius)


class ApproxMoonFinder:
    """
    Approximate moon finder using circle edge detection.
    Precomputes circle kernels for efficient processing.
    """
    # Class attributes for precomputed kernels
    _kernels = {}  # Dict mapping radius -> kernel tensor
    _min_radius = 3
    _target_size = 256
    _max_radius = _target_size // 2 - 3
    
    @classmethod
    def _create_circle_kernel(cls, radius: int, device: torch.device) -> torch.Tensor:
        """
        Create a circle kernel with 1px thick border using distance-based approach.
        Returns kernel of shape (1, 1, kernel_size, kernel_size) on specified device.
        """
        kernel_size = 2 * radius + 1
        center = radius
        
        # Create coordinate grids
        y = torch.arange(kernel_size, dtype=torch.float32, device=device)
        x = torch.arange(kernel_size, dtype=torch.float32, device=device)
        yy, xx = torch.meshgrid(y, x, indexing='ij')
        
        # Compute distance from center
        dist = torch.sqrt((yy - center) ** 2 + (xx - center) ** 2)
        
        # Set to 1 if distance is within 0.5 of radius (1px thick border)
        kernel = (torch.abs(dist - radius) < 0.5).float()
        
        # Reshape for conv2d: (1, 1, H, W)
        return kernel.unsqueeze(0).unsqueeze(0)
    
    @classmethod
    def _get_kernel(cls, radius: int, device: torch.device) -> torch.Tensor:
        """Get or create kernel for given radius."""
        if radius not in cls._kernels:
            cls._kernels[radius] = cls._create_circle_kernel(radius, device)
        # Move kernel to requested device if needed
        kernel = cls._kernels[radius]
        if kernel.device != device:
            kernel = kernel.to(device)
            cls._kernels[radius] = kernel
        return kernel
    
    @classmethod
    def find_moon_approx(cls, img: torch.Tensor) -> tuple[int, int]:
        """
        Find moon center using approximate circle edge detection.
        img: (H, W, 3) float32 [0,1] on GPU. Returns (i, j) as tuple of ints in original image space.
        """
        assert img.ndim == 3 and img.shape[2] == 3
        original_H, original_W = img.shape[0], img.shape[1]
        device = img.device
        
        # Grayscale and downscale preserving aspect ratio, then pad/crop to 512x512
        gray = img.mean(dim=2)  # (H, W)
        
        # Compute downscaled dimensions preserving aspect ratio
        # Scale factor is min(target_size / original_size) to fit within target_size
        scale = min(cls._target_size / original_H, cls._target_size / original_W)
        H_scaled = int(round(original_H * scale))
        W_scaled = int(round(original_W * scale))
        
        # Downscale preserving aspect ratio
        gray_4d = gray.unsqueeze(0).unsqueeze(0)  # (1, 1, H, W) for interpolation
        gray_scaled = torch.nn.functional.interpolate(
            gray_4d, size=(H_scaled, W_scaled), 
            mode='bilinear', align_corners=False
        ).squeeze()  # (H_scaled, W_scaled)
        
        # Pad to 512x512 (centered)
        pad_h = (cls._target_size - H_scaled) // 2
        pad_w = (cls._target_size - W_scaled) // 2
        gray_downscaled = torch.nn.functional.pad(
            gray_scaled, 
            (pad_w, cls._target_size - W_scaled - pad_w, pad_h, cls._target_size - H_scaled - pad_h),
            mode='constant', value=0.0
        )  # (512, 512)
        
        H_down, W_down = gray_downscaled.shape
        assert H_down == cls._target_size and W_down == cls._target_size
        
        # Initialize tracking variables
        best_diff = torch.full((H_down, W_down), float('-inf'), device=device, dtype=torch.float32)
        sum_prev = None  # Sum from 1 iteration ago
        sum_prev2 = None  # Sum from 2 iterations ago
        
        # Iterate over radii from min_radius to max_radius
        for radius in range(cls._min_radius, cls._max_radius + 1):
            # Get kernel for this radius
            kernel = cls._get_kernel(radius, device)
            
            # Each kernel needs padding equal to its radius to produce 512x512 output
            # This ensures: output_size = 512 + 2*radius - (2*radius+1) + 1 = 512
            # and all outputs are properly aligned (each pixel corresponds to same input location)
            padding = radius
            
            # Compute sum along circle using conv2d with per-kernel padding
            gray_input = gray_downscaled.unsqueeze(0).unsqueeze(0)  # (1, 1, H, W)
            sum_along_circle = torch.nn.functional.conv2d(
                gray_input, kernel, padding=padding
            ).squeeze()  # (H, W)
            
            # Verify output size is correct (should always be 512x512)
            assert sum_along_circle.shape == (H_down, W_down), \
                f"Output shape mismatch: expected ({H_down}, {W_down}), got {sum_along_circle.shape} for radius {radius}"
            
            # If we have sum from 2 iterations ago, compute diff
            if sum_prev2 is not None:
                # diff = sum_now - sum_2iter_before_now
                diff = sum_along_circle - sum_prev2
                # Update best diff per pixel
                best_diff = torch.maximum(best_diff, diff)
            
            # Update history: shift by one iteration
            sum_prev2 = sum_prev
            sum_prev = sum_along_circle
        
        # Find pixel with maximum diff
        flat_idx = best_diff.argmax().item()
        i_down = flat_idx // W_down
        j_down = flat_idx % W_down
        
        # Map coordinates back to original image space
        # First, subtract padding offsets to get coordinates in scaled (non-padded) space
        i_scaled = i_down - pad_h
        j_scaled = j_down - pad_w
        
        # Then scale back to original image space
        # With align_corners=False, the mapping uses half-pixel alignment:
        # output_pos = (input_pos + 0.5) * (output_size / input_size) - 0.5
        # Inverse: input_pos = (output_pos + 0.5) * (input_size / output_size) - 0.5
        i = (i_scaled + 0.5) * (original_H / H_scaled) - 0.5
        j = (j_scaled + 0.5) * (original_W / W_scaled) - 0.5
        i = int(round(i))
        j = int(round(j))
        
        if False:
            # Visualize result
            img_np = img.cpu().numpy()
            plt.figure(figsize=(12, 8))
            plt.imshow(img_np)
            plt.gca().add_patch(plt.Circle((j, i), DEBUG_RADIUS_PX, color="green", fill=True))
            plt.title(f"Moon center (approx): (i={i}, j={j})")
            plt.axis("off")
            plt.tight_layout()
            plt.show()
        
        return (i, j)
    

In [7]:
image_infos = get_image_infos()

for ii in tqdm.tqdm(image_infos, desc="Finding moon"):
    img = Image.open(ii.path)
    img_arr = torch.from_numpy(np.array(img).astype(np.float32) / 255.0).cuda()
    i0, j0 = ApproxMoonFinder.find_moon_approx(img_arr)
    i, j, radius = find_moon(img_arr, i0, j0)
    ii.moon = (i, j, radius)
    ii.moon_info_origin = MoonInfoOrigin.DIRECT

Finding moon: 100%|██████████| 84/84 [01:02<00:00,  1.35it/s]


In [8]:
exposure_groups = collections.defaultdict(list)
for ii in image_infos:
    exposure_groups[ii.exposure_time].append(ii)
for exposure_time in sorted(exposure_groups.keys()):
    group = exposure_groups[exposure_time]
    radii = [ii.moon[2] for ii in group]
    brightnesses = [ii.avg_brightness for ii in group]
    print(f'{exposure_time=:.5f} {len(group)=} {np.mean(radii)=:.2f} {np.std(radii)=:.2f} {np.mean(brightnesses)=:.6f} {np.std(brightnesses)=:.8f}')


exposure_time=0.00025 len(group)=6 np.mean(radii)=316.05 np.std(radii)=0.03 np.mean(brightnesses)=0.000547 np.std(brightnesses)=0.00000920
exposure_time=0.00050 len(group)=7 np.mean(radii)=316.03 np.std(radii)=0.03 np.mean(brightnesses)=0.000998 np.std(brightnesses)=0.00001530
exposure_time=0.00100 len(group)=4 np.mean(radii)=315.99 np.std(radii)=0.04 np.mean(brightnesses)=0.001850 np.std(brightnesses)=0.00001335
exposure_time=0.00156 len(group)=4 np.mean(radii)=315.84 np.std(radii)=0.02 np.mean(brightnesses)=0.002785 np.std(brightnesses)=0.00000427
exposure_time=0.00200 len(group)=4 np.mean(radii)=315.81 np.std(radii)=0.07 np.mean(brightnesses)=0.003390 np.std(brightnesses)=0.00001108
exposure_time=0.00400 len(group)=4 np.mean(radii)=315.59 np.std(radii)=0.06 np.mean(brightnesses)=0.005931 np.std(brightnesses)=0.00000360
exposure_time=0.00800 len(group)=4 np.mean(radii)=315.31 np.std(radii)=0.18 np.mean(brightnesses)=0.009817 np.std(brightnesses)=0.00000499
exposure_time=0.01667 len(g

In [9]:
RADIUS_STD_THRESHOLD = 1.0
MIN_GROUP_ELMS = 3

def radius(ii):
    return ii.moon[2]

prev_avg_radius = None
sorted_exposure_times = sorted(exposure_groups.keys())

for exposure_time in sorted_exposure_times:
    group = list(exposure_groups[exposure_time])
    assert len(group) >= MIN_GROUP_ELMS
    radii = [radius(ii) for ii in group]
    mean_r = np.mean(radii)
    std_r = np.std(radii)
    mean_r_ok = prev_avg_radius is None or abs(mean_r - prev_avg_radius) <= 0.1 * prev_avg_radius
    if std_r <= RADIUS_STD_THRESHOLD and mean_r_ok:
        prev_avg_radius = mean_r
    else:
        assert prev_avg_radius is not None, 'Failure in first group, it was expected to work allways'
        subgroup = [ii for ii in group if abs(radius(ii) - prev_avg_radius) <= 0.1 * prev_avg_radius]
        if len(subgroup) < MIN_GROUP_ELMS:
            subgroup = []
        else:
            mean_r = np.mean([radius(ii) for ii in subgroup])
            std_r = np.std([radius(ii) for ii in subgroup])
            if std_r > RADIUS_STD_THRESHOLD:
                subgroup = []
            else:
                prev_avg_radius = mean_r
        for ii in group:
            if ii not in subgroup:
                ii.moon = None
                ii.moon_info_origin = None

In [10]:
# do linear interpolation: fill moon (i, j, radius) for image_infos where moon is None,
# using linear (i, j) vs time and radius from group average or last group with moons.

# 1. Build linear model (i, j) = f(timestamp) from image_infos that have moon is not None
pts_with_moon = [(ii.timestamp, ii.moon[0], ii.moon[1]) for ii in image_infos if ii.moon is not None]
assert len(pts_with_moon) >= 2, "Need at least 2 points with moon to fit linear model"
t_arr = np.array([p[0] for p in pts_with_moon], dtype=np.float64)
i_arr = np.array([p[1] for p in pts_with_moon], dtype=np.float64)
j_arr = np.array([p[2] for p in pts_with_moon], dtype=np.float64)
# i = a_i * t + b_i, j = a_j * t + b_j
(a_i, b_i) = np.polyfit(t_arr, i_arr, 1)
(a_j, b_j) = np.polyfit(t_arr, j_arr, 1)

def interpolate_moon_at_time(t: float) -> tuple[float, float]:
    return (float(a_i * t + b_i), float(a_j * t + b_j))

# 2. Iterate exposure groups in ascending exposure time; fill None moons and track radius
last_avg_radius = None
for exposure_time in sorted(exposure_groups.keys()):
    group = exposure_groups[exposure_time]
    with_moon = [ii for ii in group if ii.moon is not None]
    if with_moon:
        avg_radius = float(np.mean([ii.moon[2] for ii in with_moon]))
        last_avg_radius = avg_radius
    else:
        assert last_avg_radius is not None
        avg_radius = last_avg_radius
    for ii in group:
        if ii.moon is None:
            i_pred, j_pred = interpolate_moon_at_time(ii.timestamp)
            ii.moon = (i_pred, j_pred, avg_radius)
            ii.moon_info_origin = MoonInfoOrigin.INTERPOLATED

In [11]:
for exposure_time in sorted(exposure_groups.keys()):
    group = exposure_groups[exposure_time]
    radii = [ii.moon[2] for ii in group]
    brightnesses = [ii.avg_brightness for ii in group]
    print(f'{exposure_time=:.5f} {len(group)=} {np.mean(radii)=:.2f} {np.std(radii)=:.2f} {np.mean(brightnesses)=:.6f} {np.std(brightnesses)=:.8f}')

exposure_time=0.00025 len(group)=6 np.mean(radii)=316.05 np.std(radii)=0.03 np.mean(brightnesses)=0.000547 np.std(brightnesses)=0.00000920
exposure_time=0.00050 len(group)=7 np.mean(radii)=316.03 np.std(radii)=0.03 np.mean(brightnesses)=0.000998 np.std(brightnesses)=0.00001530
exposure_time=0.00100 len(group)=4 np.mean(radii)=315.99 np.std(radii)=0.04 np.mean(brightnesses)=0.001850 np.std(brightnesses)=0.00001335
exposure_time=0.00156 len(group)=4 np.mean(radii)=315.84 np.std(radii)=0.02 np.mean(brightnesses)=0.002785 np.std(brightnesses)=0.00000427
exposure_time=0.00200 len(group)=4 np.mean(radii)=315.81 np.std(radii)=0.07 np.mean(brightnesses)=0.003390 np.std(brightnesses)=0.00001108
exposure_time=0.00400 len(group)=4 np.mean(radii)=315.59 np.std(radii)=0.06 np.mean(brightnesses)=0.005931 np.std(brightnesses)=0.00000360
exposure_time=0.00800 len(group)=4 np.mean(radii)=315.31 np.std(radii)=0.18 np.mean(brightnesses)=0.009817 np.std(brightnesses)=0.00000499
exposure_time=0.01667 len(g

In [12]:
# estimate uncertainty of moon position: residual of DIRECT positions vs linear model, then max of std_i, std_j
residuals_i = []
residuals_j = []
for ii in image_infos:
    if ii.moon_info_origin != MoonInfoOrigin.DIRECT:
        continue
    i_pred, j_pred = interpolate_moon_at_time(ii.timestamp)
    residuals_i.append(ii.moon[0] - i_pred)
    residuals_j.append(ii.moon[1] - j_pred)
residuals_i = np.array(residuals_i)
residuals_j = np.array(residuals_j)
std_i = float(np.std(residuals_i))
std_j = float(np.std(residuals_j))
moon_position_uncertainty_px = max(std_i, std_j)
print(f'moon_position_uncertainty_px={moon_position_uncertainty_px:.2f}')

moon_position_uncertainty_px=2.10


In [13]:
# Add moon_pos_std_px to each image_info: per-group std of DIRECT radii when enough DIRECT, else moon_position_uncertainty_px
for exposure_time in sorted(exposure_groups.keys()):
    group = exposure_groups[exposure_time]
    direct_in_group = [ii for ii in group if ii.moon_info_origin == MoonInfoOrigin.DIRECT]
    std_direct = None
    if len(direct_in_group) >= MIN_GROUP_ELMS:
        std_direct = float(np.std([ii.moon[2] for ii in direct_in_group]))
    for ii in group:
        if ii.moon_info_origin == MoonInfoOrigin.DIRECT and std_direct is not None:
            ii.moon_pos_std_px = std_direct
        else:
            ii.moon_pos_std_px = moon_position_uncertainty_px

In [14]:
# Bad news about sun / moon apparent motion
# https://chatgpt.com/share/e/6996a405-7df4-800e-9c16-51b51a28ce9e

# if solar radius is 500px, then:
# 1px ~ 1.92 arcsec
# scene as a whole will move by 2344 px in 5 minutes (becays its 15deg in hour, so in pixels and in 5min we have (15*3600/1.92) * (5/60))
# moon apparent movement wrt sun: 80px in 5min
# moon radius minus sun radius: maximally 40px

In [15]:
def transform_moon_center_batched(moon_center_i: float, moon_center_j: float, ci: float, cj: float, 
                                   shift_i_t: torch.Tensor, shift_j_t: torch.Tensor, 
                                   cos_a_t: torch.Tensor, sin_a_t: torch.Tensor) -> tuple[tuple[float, float]]:
    """
    Transform moon center position using the same transform as apply_transform_batched.
    
    Args:
        moon_center_i, moon_center_j: Source moon center coordinates
        ci, cj: Image center coordinates
        shift_i_t, shift_j_t: Shift tensors of shape (N, 1, 1)
        cos_a_t, sin_a_t: Rotation cosine/sine tensors of shape (N, 1, 1)
    
    Returns:
        Tuple of tuples: ((i1, j1), (i2, j2), ..., (iN, jN)) where each (i, j) is the transformed position
    """
    # Forward transform: source -> output (inverse of the transform in apply_transform_batched)
    di_src = moon_center_i - ci  # (scalar)
    dj_src = moon_center_j - cj  # (scalar)
    # Apply inverse rotation: R^T where R is the rotation matrix in apply_transform_batched
    # Use .flatten() to ensure 1D tensor even when N=1 (where .squeeze() would make it 0D)
    cos_a_flat = cos_a_t.flatten()  # (N,)
    sin_a_flat = sin_a_t.flatten()  # (N,)
    shift_i_flat = shift_i_t.flatten()  # (N,)
    shift_j_flat = shift_j_t.flatten()  # (N,)
    di_rot = di_src * cos_a_flat - dj_src * sin_a_flat  # (N,)
    dj_rot = di_src * sin_a_flat + dj_src * cos_a_flat  # (N,)
    # Add shift and center
    i_out = di_rot + shift_i_flat + ci  # (N,)
    j_out = dj_rot + shift_j_flat + cj  # (N,)
    # Convert to tuple of tuples
    return tuple((float(i_out[k]), float(j_out[k])) for k in range(len(i_out)))

def find_common_centers_and_radii(moon_center_target: tuple[float, float], moon_radius_target: float, moon_centers_warped: tuple[tuple[float, float], ...], moon_radius_warped: float) -> tuple[tuple[float, float], float]:
    centers = list()
    radii = list()
    for center2 in moon_centers_warped:
        vector = (center2[0] - moon_center_target[0], center2[1] - moon_center_target[1])
        distance = math.sqrt(vector[0]**2 + vector[1]**2)
        if distance < 0.001:
            center = center2
            radius = max(moon_radius_target, moon_radius_warped)
        else:
            vector = (vector[0] / distance, vector[1] / distance)
            pt_target = (moon_center_target[0] - vector[0] * moon_radius_target, moon_center_target[1] - vector[1] * moon_radius_target)
            pt_warped = (center2[0] + vector[0] * moon_radius_warped, center2[1] + vector[1] * moon_radius_warped)
            center = (0.5 * (pt_target[0] + pt_warped[0]), 0.5 * (pt_target[1] + pt_warped[1]))
            radius = 0.5 * (distance + moon_radius_target + moon_radius_warped)
        centers.append(center)
        radii.append(radius)
    return centers, radii

def find_max_radii(common_centers: tuple[tuple[float, float], ...], height: int, width: int) -> float:
    radii = list()
    for center in common_centers:
        di = np.abs(center[0] - height / 2)
        dj = np.abs(center[1] - width / 2)
        ri = height / 2 - di
        rj = width / 2 - dj
        radius = min(ri, rj)
        assert radius > 10, (center, height, width)
        radii.append(radius)
    return radii

def image_to_polars(image: torch.Tensor, center: tuple[float, float], radius_min: float, radius_max: float) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Convert image region between radius_min and radius_max to polar coordinates.
    
    Args:
        image: (H, W) input image tensor
        center: (center_i, center_j) center point for polar coordinates
        radius_min: minimum radius (inner edge)
        radius_max: maximum radius (outer edge)
    
    Returns:
        (polar_image, mask) where:
        - polar_image: (output_height, output_width) unrolled polar image
          - Horizontal axis: angle (0 to 2π)
          - Vertical axis: radius (radius_max at top, radius_min at bottom)
        - mask: (output_height, output_width) boolean mask indicating valid pixels
    """
    assert image.ndim == 2, image.shape
    assert len(center) == 2, len(center)
    assert 0 < radius_min < radius_max, (radius_min, radius_max)
    
    device = image.device
    dtype = image.dtype
    H, W = image.shape
    center_i, center_j = center
    
    # Output dimensions
    output_height = int(radius_max - radius_min + 1)
    output_width = int(2 * math.pi * radius_max)
    
    # Create output coordinate grids
    # y: 0 to output_height-1 (rows, radius decreases as y increases)
    # x: 0 to output_width-1 (columns, angle increases as x increases)
    y = torch.arange(output_height, device=device, dtype=torch.float32).view(-1, 1)  # (H_out, 1)
    x = torch.arange(output_width, device=device, dtype=torch.float32).view(1, -1)    # (1, W_out)
    
    # Map output coordinates to polar coordinates
    # Radius: r = radius_max - y (top row is outer radius)
    r = radius_max - y  # (H_out, 1)
    
    # Angle: θ = 2π * x / (output_width - 1) if output_width > 1, else 0
    if output_width > 1:
        theta = 2 * math.pi * x / (output_width - 1)  # (1, W_out)
    else:
        theta = torch.zeros_like(x)  # (1, W_out)
    
    # Convert polar to Cartesian coordinates in input image space
    # i = center_i + r * sin(θ)
    # j = center_j + r * cos(θ)
    i_src = center_i + r * torch.sin(theta)  # (H_out, W_out)
    j_src = center_j + r * torch.cos(theta)  # (H_out, W_out)
    
    # Normalize coordinates for grid_sample (range [-1, 1])
    # For align_corners=True: normalized = 2 * coord / (size - 1) - 1
    j_norm = 2.0 * j_src / (W - 1) - 1.0 if W > 1 else torch.zeros_like(j_src)
    i_norm = 2.0 * i_src / (H - 1) - 1.0 if H > 1 else torch.zeros_like(i_src)
    
    # Create grid for grid_sample: (H_out, W_out, 2) where last dim is [j, i]
    grid = torch.stack([j_norm, i_norm], dim=-1)  # (H_out, W_out, 2)
    
    # Prepare image for grid_sample: (1, 1, H, W)
    img_4d = image.unsqueeze(0).unsqueeze(0)
    
    # Sample with bilinear interpolation and zero padding
    polar_image = torch.nn.functional.grid_sample(
        img_4d, grid.unsqueeze(0), mode="bilinear", padding_mode="zeros", align_corners=True
    ).squeeze(0).squeeze(0)  # (H_out, W_out)
    
    # Create mask: valid pixels are those within image bounds
    # A pixel is valid if its source coordinates are within [0, H) and [0, W)
    mask = (i_src >= 0) & (i_src < H) & (j_src >= 0) & (j_src < W)
    
    return (polar_image, mask)

def remove_lowfeq(polar_image: torch.Tensor, n_remove: int) -> torch.Tensor:
    """Per-row 1D FFT: zero first 16 frequencies, then reconstruct."""
    assert polar_image.ndim == 2, polar_image.shape
    assert polar_image.shape[0] > 64, polar_image.shape
    assert polar_image.shape[1] > 64, polar_image.shape
    W = polar_image.shape[1]
    # Real FFT along each row (many 1D FFTs): (H, W) -> (H, W//2+1) complex
    spec = torch.fft.rfft(polar_image, dim=-1)
    spec[:, :n_remove] = 0.0
    out = torch.fft.irfft(spec, n=W, dim=-1)
    return out.to(polar_image.dtype)

def discrepancy_batched_fourier(target: torch.Tensor, warped: torch.Tensor, moon_center_target: tuple[float, float], moon_radius_target: float, moon_centers_warped: tuple[tuple[float, float], ...], moon_radius_warped: float) -> torch.Tensor:
    assert target.ndim == 2, target.shape
    assert warped.ndim == 3, warped.shape
    assert target.shape == warped.shape[1:], (target.shape, warped.shape)
    assert len(moon_center_target) == 2, len(moon_center_target)
    assert len(moon_centers_warped) == len(warped), (len(moon_centers_warped), len(warped))
    for i in range(len(moon_centers_warped)):
        assert len(moon_centers_warped[i]) == 2, len(moon_centers_warped[i])
    assert moon_radius_target > 0, moon_radius_target
    assert moon_radius_warped > 0, moon_radius_warped
    common_centers, radii_min = find_common_centers_and_radii(moon_center_target, moon_radius_target, moon_centers_warped, moon_radius_warped)
    radii_max = find_max_radii(common_centers, target.shape[0], target.shape[1])
    errs = list()
    for i in range(len(common_centers)):
        target_in_polars, _ = image_to_polars(target, common_centers[i], radii_min[i] + 3, radii_max[i])
        target_in_polars = remove_lowfeq(target_in_polars, 2)
        warped_in_polars, _ = image_to_polars(warped[i], common_centers[i], radii_min[i] + 3, radii_max[i])
        warped_in_polars = remove_lowfeq(warped_in_polars, 2)
        diff = torch.abs(target_in_polars - warped_in_polars)
        denominator = torch.maximum(target_in_polars.abs(), warped_in_polars.abs()).clamp(min=0.05)
        err = (diff / denominator).mean()
        errs.append(err.item())
    return torch.tensor(errs)

def compute_gradient_with_blur(image: torch.Tensor, sigma: float = 3.0) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Compute gradient of pixel intensities using Sobel filters and apply Gaussian blur.
    
    Args:
        image: (H, W) input image tensor
        sigma: Standard deviation for Gaussian blur (default: 3.0)
    
    Returns:
        (image_grad_x, image_grad_y) tuple of gradient components, both blurred
    """
    device = image.device
    # Sobel filters for gradient computation
    sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32, device=device).view(1, 1, 3, 3)
    sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32, device=device).view(1, 1, 3, 3)
    # Add batch and channel dimensions for conv2d: (H, W) -> (1, 1, H, W)
    img_4d = image.unsqueeze(0).unsqueeze(0)
    grad_x = torch.nn.functional.conv2d(img_4d, sobel_x, padding=1).squeeze()
    grad_y = torch.nn.functional.conv2d(img_4d, sobel_y, padding=1).squeeze()
    
    # Apply Gaussian blur with specified sigma
    kernel_size = int(6 * sigma + 1)  # Ensure kernel covers most of Gaussian
    if kernel_size % 2 == 0:
        kernel_size += 1  # Make odd
    
    # Create 1D Gaussian kernel
    x = torch.arange(kernel_size, dtype=torch.float32, device=device) - kernel_size // 2
    gaussian_1d = torch.exp(-0.5 * (x / sigma) ** 2)
    gaussian_1d = gaussian_1d / gaussian_1d.sum()  # Normalize
    
    # Create 2D Gaussian kernel (separable: outer product)
    gaussian_2d = gaussian_1d.view(1, 1, kernel_size, 1) * gaussian_1d.view(1, 1, 1, kernel_size)
    gaussian_2d = gaussian_2d / gaussian_2d.sum()  # Normalize again
    
    # Apply Gaussian blur to both gradients
    grad_x_4d = grad_x.unsqueeze(0).unsqueeze(0)
    grad_y_4d = grad_y.unsqueeze(0).unsqueeze(0)
    image_grad_x = torch.nn.functional.conv2d(grad_x_4d, gaussian_2d, padding=kernel_size//2).squeeze()
    image_grad_y = torch.nn.functional.conv2d(grad_y_4d, gaussian_2d, padding=kernel_size//2).squeeze()
    
    return (image_grad_x, image_grad_y)

def find_mask(image: torch.Tensor) -> torch.Tensor:
    """Find mask of valid pixels in polar image."""
    assert image.ndim == 2, image.shape
    assert image.shape[0] > 1024, image.shape
    assert image.shape[1] > 1024, image.shape
    n_cluster = 3
    # image_grad_x, image_grad_y = compute_gradient_with_blur(image, sigma=3.0)
    di = image.shape[0] // 24
    dj = image.shape[1] // 24
    mask = torch.zeros_like(image, dtype=torch.bool)
    for inp in [image]:
        inp = inp.abs()
        for _ in range(n_cluster):
            flat_idx = inp.argmax()
            i, j = torch.unravel_index(flat_idx, inp.shape)
            i0 = max(0, i - di)
            i1 = min(inp.shape[0], i + di)
            j0 = max(0, j - dj)
            j1 = min(inp.shape[1], j + dj)
            mask[i0:i1, j0:j1] = True
            inp[i0:i1, j0:j1] = 0.0
    return mask

def discrepancy_batched_fourier2(target: torch.Tensor, warped: torch.Tensor, moon_center_target: tuple[float, float], moon_radius_target: float, moon_centers_warped: tuple[tuple[float, float], ...], moon_radius_warped: float) -> torch.Tensor:
    assert target.ndim == 2, target.shape
    assert warped.ndim == 3, warped.shape
    assert target.shape == warped.shape[1:], (target.shape, warped.shape)
    assert len(moon_center_target) == 2, len(moon_center_target)
    assert len(moon_centers_warped) == len(warped), (len(moon_centers_warped), len(warped))
    for i in range(len(moon_centers_warped)):
        assert len(moon_centers_warped[i]) == 2, len(moon_centers_warped[i])
    assert moon_radius_target > 0, moon_radius_target
    assert moon_radius_warped > 0, moon_radius_warped
    common_centers, radii_min = find_common_centers_and_radii(moon_center_target, moon_radius_target, moon_centers_warped, moon_radius_warped)
    radii_max = find_max_radii(common_centers, target.shape[0], target.shape[1])
    errs = list()
    for i in range(len(common_centers)):
        target_in_polars, _ = image_to_polars(target, common_centers[i], radii_min[i] + 3, radii_max[i])
        target_in_polars = remove_lowfeq(target_in_polars, 2)
        target_mask = find_mask(target_in_polars)
        warped_in_polars, _ = image_to_polars(warped[i], common_centers[i], radii_min[i] + 3, radii_max[i])
        warped_in_polars = remove_lowfeq(warped_in_polars, 2)
        warped_mask = find_mask(warped_in_polars)
        mask = (target_mask | warped_mask).to(torch.float32)
        assert mask.sum() > 0
        diff = torch.abs(target_in_polars - warped_in_polars)
        err = (diff * mask).sum() / mask.sum()
        errs.append(err.item())
    return torch.tensor(errs)

def polar_to_cartesian(polar_image: torch.Tensor, center: tuple[float, float], radius_min: float, radius_max: float, output_size: tuple[int, int]) -> torch.Tensor:
    """
    Convert polar image back to Cartesian coordinates (inverse of image_to_polars).
    
    Args:
        polar_image: (H_polar, W_polar) polar image tensor
        center: (center_i, center_j) center point for polar coordinates
        radius_min: minimum radius (inner edge)
        radius_max: maximum radius (outer edge)
        output_size: (H, W) output image dimensions
    
    Returns:
        cartesian_image: (H, W) reconstructed Cartesian image
    """
    assert polar_image.ndim == 2, polar_image.shape
    assert len(center) == 2, len(center)
    assert 0 < radius_min < radius_max, (radius_min, radius_max)
    assert len(output_size) == 2, len(output_size)
    
    device = polar_image.device
    dtype = polar_image.dtype
    H_polar, W_polar = polar_image.shape
    H, W = output_size
    center_i, center_j = center
    
    # Create output coordinate grids for Cartesian image
    # i: 0 to H-1 (rows)
    # j: 0 to W-1 (columns)
    i = torch.arange(H, device=device, dtype=torch.float32).view(-1, 1)  # (H, 1)
    j = torch.arange(W, device=device, dtype=torch.float32).view(1, -1)  # (1, W)
    
    # Convert Cartesian coordinates to polar coordinates
    # Compute distance from center and angle
    di = i - center_i  # (H, 1)
    dj = j - center_j  # (1, W)
    r = torch.sqrt(di * di + dj * dj)  # (H, W)
    
    # Compute angle: atan2 gives [-π, π], we need to map to [0, 2π]
    theta = torch.atan2(di, dj)  # (H, W), range [-π, π]
    # Normalize to [0, 2π]
    theta = torch.where(theta < 0, theta + 2 * math.pi, theta)  # (H, W), range [0, 2π]
    
    # Map to polar image coordinates
    # Radius: y_polar = radius_max - r (since top row is outer radius)
    y_polar = radius_max - r  # (H, W)
    
    # Angle: x_polar = theta / (2π) * (W_polar - 1) if W_polar > 1, else 0
    if W_polar > 1:
        x_polar = theta / (2 * math.pi) * (W_polar - 1)  # (H, W)
    else:
        x_polar = torch.zeros_like(theta)  # (H, W)
    
    # Normalize coordinates for grid_sample (range [-1, 1])
    # For align_corners=True: normalized = 2 * coord / (size - 1) - 1
    x_norm = 2.0 * x_polar / (W_polar - 1) - 1.0 if W_polar > 1 else torch.zeros_like(x_polar)
    y_norm = 2.0 * y_polar / (H_polar - 1) - 1.0 if H_polar > 1 else torch.zeros_like(y_polar)
    
    # Create grid for grid_sample: (H, W, 2) where last dim is [x, y] (note: x is j, y is i in grid_sample)
    grid = torch.stack([x_norm, y_norm], dim=-1)  # (H, W, 2)
    
    # Prepare polar image for grid_sample: (1, 1, H_polar, W_polar)
    polar_4d = polar_image.unsqueeze(0).unsqueeze(0)
    
    # Sample with bilinear interpolation and zero padding
    cartesian_image = torch.nn.functional.grid_sample(
        polar_4d, grid.unsqueeze(0), mode="bilinear", padding_mode="zeros", align_corners=True
    ).squeeze(0).squeeze(0)  # (H, W)
    
    # Set pixels outside [radius_min, radius_max] to zero
    mask_valid = (r >= radius_min) & (r <= radius_max)
    cartesian_image = torch.where(mask_valid, cartesian_image, torch.zeros_like(cartesian_image))
    
    return cartesian_image

def fill_bottom(polar_img: torch.Tensor, n_step: int) -> torch.Tensor:
    assert polar_img.ndim == 2, polar_img.shape
    assert 0 <= n_step < polar_img.shape[0], (n_step, polar_img.shape)
    if n_step == 0:
        return polar_img
    retval = polar_img.clone()
    mx = polar_img[-n_step:, :].amax(dim=0)
    apply = torch.ones(retval.shape[1], dtype=torch.bool, device=retval.device)
    for i in range(n_step):
        apply = apply & (retval[-i-1, :] >= mx)
        retval[-i-1, apply] = mx[apply]
    return retval

def fill_moon(img: torch.Tensor, moon_center: tuple[float, float], moon_radius: float, value: float) -> torch.Tensor:
    """Fill with value all pixels whose distance from moon_center is <= moon_radius. Returns a clone."""
    assert img.ndim == 2, img.shape
    assert len(moon_center) == 2, len(moon_center)
    assert moon_radius > 0, moon_radius
    out = img.clone()
    ci, cj = moon_center
    H, W = img.shape
    i = torch.arange(H, dtype=torch.float32, device=img.device)[:, None]
    j = torch.arange(W, dtype=torch.float32, device=img.device)[None, :]
    dist = torch.sqrt((i - ci) ** 2 + (j - cj) ** 2)
    mask = dist <= moon_radius
    out[mask] = value
    return out

def gaussian_blur(img: torch.Tensor, sigma: float) -> torch.Tensor:
    assert img.ndim == 2, img.shape
    assert sigma > 0, sigma
    sigma = int(np.ceil(sigma))
    kernel_size = min(17, 4 * sigma + 1)
    gblur = torchvision.transforms.GaussianBlur(kernel_size=kernel_size, sigma=sigma).to(img.device)
    img = gblur(img.unsqueeze(0).unsqueeze(0)).squeeze(0).squeeze(0)
    return img

def discrepancy_batched_fourier3(target: torch.Tensor, warped: torch.Tensor, moon_center_target: tuple[float, float], moon_radius_target: float, moon_centers_warped: tuple[tuple[float, float], ...], moon_radius_warped: float, batch: list[tuple[float, float, float]], blur_sigma: float, best_setup: None | tuple[float, float, float, torch.Tensor, torch.Tensor]) -> torch.Tensor:
    assert target.ndim == 2, target.shape
    assert warped.ndim == 3, warped.shape
    assert target.shape == warped.shape[1:], (target.shape, warped.shape)
    assert len(moon_center_target) == 2, len(moon_center_target)
    assert len(moon_centers_warped) == len(warped), (len(moon_centers_warped), len(warped))
    for i in range(len(moon_centers_warped)):
        assert len(moon_centers_warped[i]) == 2, len(moon_centers_warped[i])
    assert moon_radius_target > 0, moon_radius_target
    assert moon_radius_warped > 0, moon_radius_warped
    assert blur_sigma >= 0, blur_sigma
    assert len(batch) == len(warped), (len(batch), len(warped))
    # make list of all images
    list_of_all = [(target, moon_center_target, moon_radius_target)]
    for i in range(len(moon_centers_warped)):
        list_of_all.append((warped[i], moon_centers_warped[i], moon_radius_warped))
    list_of_all_processed = []
    for img, moon_center, moon_radius in list_of_all:
        radius_max = min(moon_center[0], img.shape[0] - moon_center[0], moon_center[1], img.shape[1] - moon_center[1])
        assert 32 < moon_radius < radius_max - 32
        polar_img, _ = image_to_polars(img, moon_center, moon_radius, radius_max)
        polar_img = fill_bottom(polar_img, 4)
        polar_img = remove_lowfeq(polar_img, 16)
        img = polar_to_cartesian(polar_img, moon_center, moon_radius, radius_max, (img.shape[0], img.shape[1]))
        if blur_sigma > 0:
            img = gaussian_blur(img, blur_sigma)
        mask = torch.ones_like(polar_img)
        mask = polar_to_cartesian(mask, moon_center, moon_radius, radius_max, (img.shape[0], img.shape[1]))
        mask = fill_moon(mask, moon_center, moon_radius, 0.0)
        list_of_all_processed.append((img, mask))
    target_img, target_mask = list_of_all_processed.pop(0)
    if best_setup is not None:
        bs_shift_i, bs_shift_j, bs_angle, bs_img, bs_mask = best_setup
        batch.append((bs_shift_i, bs_shift_j, bs_angle))
        list_of_all_processed.append((bs_img, bs_mask))
    assert len(list_of_all_processed) == len(batch)
    while len(list_of_all_processed) >= 2:
        shift_i_0, shift_j_0, angle_0 = batch.pop()
        shift_i_1, shift_j_1, angle_1 = batch.pop()
        warped_img_0, warped_mask_0 = list_of_all_processed.pop()
        warped_img_1, warped_mask_1 = list_of_all_processed.pop()
        mask = target_mask * warped_mask_0 * warped_mask_1
        diff_0 = torch.abs(target_img - warped_img_0)
        diff_0 = ((diff_0 * mask).sum() / mask.sum()).item()
        diff_1 = torch.abs(target_img - warped_img_1)
        diff_1 = ((diff_1 * mask).sum() / mask.sum()).item()
        if diff_0 < diff_1:
            batch.append((shift_i_0, shift_j_0, angle_0))
            list_of_all_processed.append((warped_img_0, warped_mask_0))
        else:
            batch.append((shift_i_1, shift_j_1, angle_1))
            list_of_all_processed.append((warped_img_1, warped_mask_1))
    shift_i, shift_j, angle = batch.pop()
    warped_img, warped_mask = list_of_all_processed.pop()
    return shift_i, shift_j, angle, warped_img, warped_mask

In [25]:
def register_equal_exposure(image_info0, image_info1):
    """
    Find (shift_i, shift_j, rotation_deg) to align the second image to the first.
    rotation_deg: degrees counterclockwise. shifts in pixels.
    Uses iterative 5x5x5 grid search on GPU; grid can reduce to 5x5x1 or 1x1x5 when one dimension converges.
    """
    device = torch.device("cuda")
    # Load images: grayscale float32 [0,1] on GPU
    def load_grayscale(ii):
        with Image.open(ii.path) as img:
            arr = np.array(img).astype(np.float32) / 255.0
        if arr.ndim == 3:
            arr = arr.mean(axis=2)
        return torch.from_numpy(arr).to(device=device, dtype=torch.float32)

    g0 = load_grayscale(image_info0)   # (H, W)
    g1 = load_grayscale(image_info1)   # (H, W)
    H, W = g0.shape
    assert g1.shape == (H, W)

    # Moon data (center_i, center_j, radius)
    moon0 = image_info0.moon
    moon1 = image_info1.moon
    r0 = moon0[2]
    r1 = moon1[2]
    moon_radius_avg = (r0 + r1) / 2.0

    # Uncertainty and sun drift
    u0 = image_info0.moon_pos_std_px if image_info0.moon_pos_std_px is not None else 2.0
    u1 = image_info1.moon_pos_std_px if image_info1.moon_pos_std_px is not None else 2.0
    sun_drift_per_sec = 0.001 * moon_radius_avg
    dt_sec = abs(image_info1.timestamp - image_info0.timestamp)
    possible_sun_drift = dt_sec * sun_drift_per_sec
    initial_shift_half = 5.0 * (u0 + u1) + possible_sun_drift
    # double it to stay safe
    initial_shift_half = 2.0 * (initial_shift_half + 3.0)

    # Rotation center = image center
    ci, cj = H / 2.0, W / 2.0
    r_border = max(H, W) # max distance from center to corner (in fact its too much, but it works)

    # Precompute output pixel coords (1, H, W) for batched broadcast with (N, 1, 1)
    ii = torch.arange(H, device=device, dtype=torch.float32).view(-1, 1).expand(H, W).unsqueeze(0)
    jj = torch.arange(W, device=device, dtype=torch.float32).view(1, -1).expand(H, W).unsqueeze(0)

    def apply_transform_batched(img, shift_i_t, shift_j_t, cos_a_t, sin_a_t):
        # shift_* (N,1,1), cos_a_t, sin_a_t (N,1,1). Output (N, H, W).
        di = ii - ci - shift_i_t   # (N, H, W)
        dj = jj - cj - shift_j_t
        i_src = di * cos_a_t + dj * sin_a_t + ci
        j_src = -di * sin_a_t + dj * cos_a_t + cj
        j_norm = 2.0 * j_src / (W - 1) - 1.0 if W > 1 else torch.zeros_like(j_src)
        i_norm = 2.0 * i_src / (H - 1) - 1.0 if H > 1 else torch.zeros_like(i_src)
        grid = torch.stack([j_norm, i_norm], dim=-1)  # (N, H, W, 2)
        img_4d = img.unsqueeze(0).unsqueeze(1)  # (1, 1, H, W)
        img_4d = img_4d.expand(grid.shape[0], 1, H, W)
        out = torch.nn.functional.grid_sample(
            img_4d, grid, mode="bilinear", padding_mode="zeros", align_corners=True
        )
        return out.squeeze(1)  # (N, H, W)

    def discrepancy_batched(target, warped):
        # target (H,W), warped (N, H, W). Return (N,) errors.
        both_lt_08 = (target < 0.8) & (warped < 0.8)
        any_gt_0 = (target > 0) | (warped > 0)
        mask = both_lt_08 & any_gt_0
        n = mask.sum(dim=(1, 2)).float().clamp(min=1.0)
        denominator = torch.maximum(target.abs(), warped.abs()).clamp(min=0.01)
        return (torch.abs(target - warped) * mask.float() / denominator).sum(dim=(1, 2)) / n

    # Grid search state
    best_shift_i = 0.0
    best_shift_j = 0.0
    best_angle = 0.0
    step_shift = initial_shift_half / 2.0   # 5 points over ±initial_shift_half
    step_angle = 5.0   # 5 points from -10 to +10 deg
    refine_shift = True
    refine_angle = True

    best_setup = None
    while refine_shift or refine_angle:
        if step_shift < 0.1:
            refine_shift = False
        step_angle_in_px = step_angle * r_border * (math.pi / 180.0)
        if step_angle_in_px < 0.1:
            refine_angle = False
        if step_shift < 2 and step_angle_in_px < 2:
            blur_sigma = 0.0
        else:
            blur_sigma = min(8, max(step_angle_in_px, step_shift))

        shift_i_vals = (
            [best_shift_i] if not refine_shift
            else [best_shift_i + step_shift * (k - 2) for k in range(5)]
        )
        shift_j_vals = (
            [best_shift_j] if not refine_shift
            else [best_shift_j + step_shift * (k - 2) for k in range(5)]
        )
        angle_vals = (
            [best_angle] if not refine_angle
            else [best_angle + step_angle * (k - 2) for k in range(5)]
        )

        triples = [(si, sj, a) for si in shift_i_vals for sj in shift_j_vals for a in angle_vals]
        # Process in small batches to avoid OOM (full grid would be 125 x H x W)
        GRID_BATCH_SIZE = 8  # decrease to 2 or 1 if still OOM on large images
        for start in range(0, len(triples), GRID_BATCH_SIZE):
            batch = triples[start : start + GRID_BATCH_SIZE]
            N = len(batch)
            shift_i_t = torch.tensor([t[0] for t in batch], device=device, dtype=torch.float32).view(N, 1, 1)
            shift_j_t = torch.tensor([t[1] for t in batch], device=device, dtype=torch.float32).view(N, 1, 1)
            angles_rad = torch.tensor([math.radians(-t[2]) for t in batch], device=device, dtype=torch.float32)
            cos_a_t = torch.cos(angles_rad).view(N, 1, 1)
            sin_a_t = torch.sin(angles_rad).view(N, 1, 1)
            warped = apply_transform_batched(g1.clone(), shift_i_t, shift_j_t, cos_a_t, sin_a_t)
            moon_centers_warped = transform_moon_center_batched(moon1[0], moon1[1], ci, cj, shift_i_t, shift_j_t, cos_a_t, sin_a_t)
            best_setup = discrepancy_batched_fourier3(g0.clone(), warped, moon0[:2], r0, moon_centers_warped, r1, batch, blur_sigma, best_setup)
        best_shift_i, best_shift_j, best_angle, _, _ = best_setup
        is_corner = len(shift_i_vals) > 2 and (best_shift_i in [shift_i_vals[0], shift_i_vals[-1]] or best_shift_j in [shift_j_vals[0], shift_j_vals[-1]])
        step_shift = step_shift / 2.0 if refine_shift and not is_corner else step_shift
        is_corner = len(angle_vals) > 2 and best_angle in [angle_vals[0], angle_vals[-1]]
        step_angle = step_angle / 2.0 if refine_angle and not is_corner else step_angle

    return (float(best_shift_i), float(best_shift_j), float(best_angle))

In [26]:
# Apply register_equal_exposure to two randomly chosen images from each exposure group
def load_grayscale_for_debug(ii):
    with Image.open(ii.path) as img:
        arr = np.array(img).astype(np.float32) / 255.0
    if arr.ndim == 3:
        arr = arr.mean(axis=2)
    return torch.from_numpy(arr).cuda().to(torch.float32)

def apply_transform_single(img, shift_i, shift_j, angle_deg, device):
    H, W = img.shape
    ci, cj = H / 2.0, W / 2.0
    angle_rad = math.radians(-angle_deg)
    cos_a, sin_a = math.cos(angle_rad), math.sin(angle_rad)
    ii = torch.arange(H, device=device, dtype=torch.float32).view(-1, 1).expand(H, W)
    jj = torch.arange(W, device=device, dtype=torch.float32).view(1, -1).expand(H, W)
    di = ii - ci - shift_i
    dj = jj - cj - shift_j
    i_src = di * cos_a + dj * sin_a + ci
    j_src = -di * sin_a + dj * cos_a + cj
    j_norm = 2.0 * j_src / (W - 1) - 1.0 if W > 1 else torch.zeros_like(j_src)
    i_norm = 2.0 * i_src / (H - 1) - 1.0 if H > 1 else torch.zeros_like(i_src)
    grid = torch.stack([j_norm, i_norm], dim=-1).unsqueeze(0)
    out = torch.nn.functional.grid_sample(
        img.unsqueeze(0).unsqueeze(0), grid, mode="bilinear", padding_mode="zeros", align_corners=True
    )
    return out.squeeze(0).squeeze(0)

for exposure_time in sorted(exposure_groups.keys()):
    group = exposure_groups[exposure_time]
    if len(group) < 2:
        print(f"exposure_time={exposure_time:.5f} skip (group size {len(group)} < 2)")
        continue
    ii0, ii1 = random.sample(group, 2)
    shift_i, shift_j, rotation = register_equal_exposure(ii0, ii1)
    print(f"exposure_time={exposure_time:.5f} (n={len(group)}) "
          f"shift_i={shift_i:.4f} shift_j={shift_j:.4f} rotation_deg={rotation:.4f} "
          f"# {ii0.path.name} vs {ii1.path.name}")
    if False:
        # Debug: two crops side by side (crop = moon center, size 2.4 * moon radius)
        g0 = load_grayscale_for_debug(ii0)
        g1 = load_grayscale_for_debug(ii1)
        g1_aligned = apply_transform_single(g1, shift_i, shift_j, rotation, g0.device)
        moon_i, moon_j, moon_r = ii0.moon[0], ii0.moon[1], ii0.moon[2]
        half = 1.2 * moon_r
        i_lo = max(0, int(moon_i - half))
        i_hi = min(g0.shape[0], int(moon_i + half))
        j_lo = max(0, int(moon_j - half))
        j_hi = min(g0.shape[1], int(moon_j + half))
        crop0 = g0[i_lo:i_hi, j_lo:j_hi].cpu().numpy()
        crop1_aligned = g1_aligned[i_lo:i_hi, j_lo:j_hi].cpu().numpy()
        diff = np.abs(crop0.astype(np.float64) - crop1_aligned.astype(np.float64))
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
        ax1.imshow(crop0, cmap="gray", vmin=0, vmax=1)
        ax1.set_title("First image (crop)")
        ax1.axis("off")
        ax2.imshow(diff, cmap="gray", vmin=0, vmax=1)
        ax2.set_title("|first - aligned second|")
        ax2.axis("off")
        plt.suptitle(f"exposure_time={exposure_time:.5f}")
        plt.tight_layout()
        plt.show()

exposure_time=0.00025 (n=6) shift_i=1.5266 shift_j=-2.5079 rotation_deg=0.0000 # img_0149_53655849382_o.jpg vs img_0150_53657076894_o.jpg
exposure_time=0.00050 (n=7) shift_i=-7.8872 shift_j=4.8826 rotation_deg=0.1562 # img_0156_53656724746_o.jpg vs img_0144_53657077009_o.jpg
exposure_time=0.00100 (n=4) shift_i=0.6430 shift_j=-0.5144 rotation_deg=0.0000 # img_0160_53656945918_o.jpg vs img_0157_53656945928_o.jpg
exposure_time=0.00156 (n=4) shift_i=-0.1074 shift_j=-0.1074 rotation_deg=0.0000 # img_0164_53656724611_o.jpg vs img_0163_53655849047_o.jpg
exposure_time=0.00200 (n=4) shift_i=0.7348 shift_j=-0.2449 rotation_deg=0.0000 # img_0167_53656945823_o.jpg vs img_0166_53655849057_o.jpg
exposure_time=0.00400 (n=4) shift_i=-0.1271 shift_j=-0.7628 rotation_deg=0.0000 # img_0169_53657076614_o.jpg vs img_0171_53656724456_o.jpg
exposure_time=0.00800 (n=4) shift_i=-3.6116 shift_j=4.5145 rotation_deg=0.0000 # img_0173_53656945698_o.jpg vs img_0176_53656724276_o.jpg
exposure_time=0.01667 (n=5) shif

In [27]:
# Verify registration
# For each exposure group: compute registration for all pairs in both directions (A->B and B->A),
# then run two checks: (1) A->B + B->A should be zero; (2) (A->B) composed with (B->C) should equal (A->C).

import itertools


def compose_transforms(s1_i, s1_j, rot1_deg, s2_i, s2_j, rot2_deg):
    """Compose transform (s1, rot1) then (s2, rot2). Returns (s_i, s_j, rot_deg).
    Each transform is (shift_i, shift_j, rotation_deg) mapping frame A -> B for sampling.
    (A->B) composed with (B->C) gives (A->C).
    """
    theta1_rad = math.radians(rot1_deg)
    cos1, sin1 = math.cos(theta1_rad), math.sin(theta1_rad)
    # R(θ1) * (s2_i, s2_j) counterclockwise
    s_rot_i = cos1 * s2_i - sin1 * s2_j
    s_rot_j = sin1 * s2_i + cos1 * s2_j
    return (s1_i + s_rot_i, s1_j + s_rot_j, rot1_deg + rot2_deg)


reg = {}  # key: (exposure_time, i, j) -> (shift_i, shift_j, rotation_deg) for transform from group[i] to group[j]

for exposure_time in sorted(exposure_groups.keys()):
    print(f"Doing check for exposure time {exposure_time}")
    group = list(exposure_groups[exposure_time])
    n = len(group)
    if n < 2:
        print(f"  Skip (group size {n} < 2)")
        continue

    # Build pairwise registration tasks for this group only
    tasks = [(i, j) for i, j in itertools.permutations(range(n), 2)]
    for i, j in tqdm.tqdm(tasks, desc="Registration pairs"):
        key = (exposure_time, i, j)
        shift_i, shift_j, rotation = register_equal_exposure(group[i], group[j])
        reg[key] = (shift_i, shift_j, rotation)

    # Check 1 for this group: reg(a->b) + reg(b->a) should be (0,0,0)
    res_i, res_j, res_rot = [], [], []
    for a, b in itertools.combinations(range(n), 2):
        rab = reg[(exposure_time, a, b)]
        rba = reg[(exposure_time, b, a)]
        res_i.append(rab[0] + rba[0])
        res_j.append(rab[1] + rba[1])
        res_rot.append(rab[2] + rba[2])
    res_i, res_j, res_rot = np.array(res_i), np.array(res_j), np.array(res_rot)
    ijmean = 0.5 * float((np.mean(np.abs(res_i)) + np.mean(np.abs(res_j))))
    ijmax = float(max(np.max(np.abs(res_i)), np.max(np.abs(res_j))))
    rotmean = float(np.mean(np.abs(res_rot)))
    rotmax = float(np.max(np.abs(res_rot)))
    print(f"  Check 1 (reg(A->B) + reg(B->A) = 0):  ij  mean = {ijmean:.4f} max = {ijmax:.4f}  rot  mean = {rotmean:.4f} max = {rotmax:.4f}")

    # Check 2 for this group: (A->B) composed with (B->C) should equal (A->C)
    if n >= 3:
        tri_i, tri_j, tri_rot = [], [], []
        for a, b, c in itertools.permutations(range(n), 3):
            rab = reg[(exposure_time, a, b)]
            rbc = reg[(exposure_time, b, c)]
            rac = reg[(exposure_time, a, c)]
            composed = compose_transforms(rab[0], rab[1], rab[2], rbc[0], rbc[1], rbc[2])
            tri_i.append(composed[0] - rac[0])
            tri_j.append(composed[1] - rac[1])
            tri_rot.append(composed[2] - rac[2])
        tri_i, tri_j, tri_rot = np.array(tri_i), np.array(tri_j), np.array(tri_rot)
        ijmean = 0.5 * float((np.mean(np.abs(tri_i)) + np.mean(np.abs(tri_j))))
        ijmax = float(max(np.max(np.abs(tri_i)), np.max(np.abs(tri_j))))
        rotmean = float(np.mean(np.abs(tri_rot)))
        rotmax = float(np.max(np.abs(tri_rot)))
        print(f"  Check 2 ((A->B)+(B->C) = (A->C)):  ij  mean = {ijmean:.4f} max = {ijmax:.4f}  rot  mean = {rotmean:.4f} max = {rotmax:.4f}")

Doing check for exposure time 0.00025


Registration pairs: 100%|██████████| 30/30 [10:22<00:00, 20.75s/it]


  Check 1 (reg(A->B) + reg(B->A) = 0):  ij  mean = 0.1741 max = 0.5102  rot  mean = 0.0130 max = 0.0781
  Check 2 ((A->B)+(B->C) = (A->C)):  ij  mean = 0.2125 max = 0.7301  rot  mean = 0.0247 max = 0.0977
Doing check for exposure time 0.0005


Registration pairs: 100%|██████████| 42/42 [13:21<00:00, 19.09s/it]


  Check 1 (reg(A->B) + reg(B->A) = 0):  ij  mean = 0.1268 max = 0.7152  rot  mean = 0.0019 max = 0.0195
  Check 2 ((A->B)+(B->C) = (A->C)):  ij  mean = 0.1618 max = 0.7149  rot  mean = 0.0028 max = 0.0391
Doing check for exposure time 0.001


...

  Check 1 (reg(A->B) + reg(B->A) = 0):  ij  mean = 0.0082 max = 0.1643  rot  mean = 0.0000 max = 0.0000
  Check 2 ((A->B)+(B->C) = (A->C)):  ij  mean = 0.1614 max = 0.3355  rot  mean = 0.0000 max = 0.0000
Doing check for exposure time 1.0


Registration pairs: 100%|██████████| 72/72 [29:05<00:00, 24.25s/it]


  Check 1 (reg(A->B) + reg(B->A) = 0):  ij  mean = 0.3149 max = 2.3160  rot  mean = 0.0065 max = 0.0781
  Check 2 ((A->B)+(B->C) = (A->C)):  ij  mean = 0.4428 max = 3.3670  rot  mean = 0.0184 max = 0.1562
Doing check for exposure time 2.0


Registration pairs: 100%|██████████| 12/12 [04:45<00:00, 23.81s/it]

  Check 1 (reg(A->B) + reg(B->A) = 0):  ij  mean = 0.1361 max = 0.8170  rot  mean = 0.0000 max = 0.0000
  Check 2 ((A->B)+(B->C) = (A->C)):  ij  mean = 0.3917 max = 0.4419  rot  mean = 0.0000 max = 0.0000


In [28]:
if False:
    # Two-image semitransparent debug: same crop region (from first image), both overlaid
    PATH1 = "/home/slavik/e202602_eclipse/data/img_0148_53656946138_o.jpg"
    PATH2 = "/home/slavik/e202602_eclipse/data/img_0149_53655849382_o.jpg"
    
    img1 = Image.open(PATH1)
    img2 = Image.open(PATH2)
    arr1 = np.array(img1).astype(np.float32) / 255.0
    arr2 = np.array(img2).astype(np.float32) / 255.0
    H, W = arr1.shape[0], arr1.shape[1]
    
    t1 = torch.from_numpy(arr1).cuda()
    t2 = torch.from_numpy(arr2).cuda()
    i1, j1, r1 = find_moon(t1, H / 2, W / 2)
    i2, j2, r2 = find_moon(t2, H / 2, W / 2)
    
    # Crop region from first image (same as in find_moon debug)
    half_crop = int(round(1.1 * r1))
    i_min = int(round(i1 - half_crop))
    i_max = int(round(i1 + half_crop))
    j_min = int(round(j1 - half_crop))
    j_max = int(round(j1 + half_crop))
    # Clamp to image bounds so same region works for both
    i_min = max(0, i_min)
    i_max = min(H - 1, i_max)
    j_min = max(0, j_min)
    j_max = min(W - 1, j_max)

    crop1 = arr1[i_min : i_max + 1, j_min : j_max + 1]
    crop2 = arr2[i_min : i_max + 1, j_min : j_max + 1]
    blended = 0.5 * crop1 + 0.5 * crop2
    
    # Draw same debug as find_moon: green circle at center + 36 border points per moon
    plt.figure(figsize=(12, 12))
    plt.imshow(blended)
    
    # Moon 1 (first image) in crop coords
    c1_j = j1 - j_min
    c1_i = i1 - i_min
    plt.gca().add_patch(plt.Circle((c1_j, c1_i), DEBUG_RADIUS_PX, color="green", fill=True))
    for k in range(36):
        angle = 2 * math.pi * k / 36
        bj = c1_j + r1 * math.cos(angle)
        bi = c1_i + r1 * math.sin(angle)
        plt.plot(bj, bi, "g.", markersize=1)
    
    # Moon 2 (second image) in crop coords
    c2_j = j2 - j_min
    c2_i = i2 - i_min
    plt.gca().add_patch(plt.Circle((c2_j, c2_i), DEBUG_RADIUS_PX, color="green", fill=True))
    for k in range(36):
        angle = 2 * math.pi * k / 36
        bj = c2_j + r2 * math.cos(angle)
        bi = c2_i + r2 * math.sin(angle)
        plt.plot(bj, bi, "g.", markersize=1)
    
    plt.title("Two images overlaid (same crop); green = moon centers and radii")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

In [29]:
if False:
    # Debug: exposure 0.00025 — find worst Check 2 triplet and plot crops for AB, BC, AC
    import itertools
    device = torch.device("cuda")
    target_exposure = 0.00025
    # Resolve key (float may be 0.00025 or 0.00025000000000000001)
    exp_key = next((k for k in exposure_groups.keys() if abs(k - target_exposure) < 1e-9), None)
    assert exp_key is not None, f"No exposure group for {target_exposure}"
    group = list(exposure_groups[exp_key])
    n = len(group)
    assert n >= 3

    # Run pairwise registrations for this group (same as cell 17)
    reg_debug = {}
    for i, j in tqdm.tqdm([(i, j) for i, j in itertools.permutations(range(n), 2)], desc="Registration pairs"):
        shift_i, shift_j, rotation = register_equal_exposure(group[i], group[j])
        reg_debug[(i, j)] = (shift_i, shift_j, rotation)

    # Find triple (a,b,c) with worst Check 2 residual (max over i, j, rot)
    worst_score = -1.0
    worst_triple = None
    for a, b, c in itertools.permutations(range(n), 3):
        rab = reg_debug[(a, b)]
        rbc = reg_debug[(b, c)]
        rac = reg_debug[(a, c)]
        composed = compose_transforms(rab[0], rab[1], rab[2], rbc[0], rbc[1], rbc[2])
        res_i = abs(composed[0] - rac[0])
        res_j = abs(composed[1] - rac[1])
        res_rot = abs(composed[2] - rac[2])
        score = max(res_i, res_j, res_rot)
        if score > worst_score:
            worst_score = score
            worst_triple = (a, b, c)
    a, b, c = worst_triple
    print(f"Worst Check 2 triple: indices ({a}, {b}, {c}), max residual = {worst_score:.4f}")
    print(f"  A={group[a].path.name}  B={group[b].path.name}  C={group[c].path.name}")

    def fill_moon_blue_np(arr, moon_i, moon_j, fill_radius):
        """arr (H,W) or (H,W,3) float [0,1]. Fill moon circle with blue (0,0,1). Returns RGB (H,W,3)."""
        if arr.ndim == 2:
            arr = np.stack([arr, arr, arr], axis=-1)
        H, W = arr.shape[0], arr.shape[1]
        y = np.arange(H, dtype=np.float32)[:, None] - moon_i
        x = np.arange(W, dtype=np.float32)[None, :] - moon_j
        r = np.sqrt(x*x + y*y)
        mask = r <= fill_radius
        out = arr.copy()
        out[mask, 0] = 0.0
        out[mask, 1] = 0.0
        out[mask, 2] = 1.0
        return out

    pairs = [(a, b), (b, c), (a, c)]
    for (i, j) in pairs:
        ii1, ii2 = group[i], group[j]
        moon1 = ii1.moon
        mi, mj, r = moon1[0], moon1[1], moon1[2]
        half = 1.2 * r
        i_lo = max(0, int(mi - half))
        i_hi = min(ii1.height, int(mi + half))
        j_lo = max(0, int(mj - half))
        j_hi = min(ii1.width, int(mj + half))

        g1 = load_grayscale_for_debug(ii1)
        g2 = load_grayscale_for_debug(ii2)
        shift_i, shift_j, rotation = reg_debug[(i, j)]
        g2_aligned = apply_transform_single(g2, shift_i, shift_j, rotation, device)

        arr1 = g1.cpu().numpy()
        arr2 = g2.cpu().numpy()
        arr2_aligned = g2_aligned.cpu().numpy()

        crop1 = arr1[i_lo:i_hi, j_lo:j_hi]
        crop2 = arr2[i_lo:i_hi, j_lo:j_hi]
        crop2_aligned = arr2_aligned[i_lo:i_hi, j_lo:j_hi]

        # 1) Both images as they are
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
        ax1.imshow(crop1, cmap="gray", vmin=0, vmax=1)
        ax1.set_title(f"Pair ({i},{j}): first (idx {i})")
        ax1.axis("off")
        ax2.imshow(crop2, cmap="gray", vmin=0, vmax=1)
        ax2.set_title(f"second (idx {j})")
        ax2.axis("off")
        plt.suptitle(f"Pair {ii1.path.name} / {ii2.path.name} — as-is")
        plt.tight_layout()
        plt.show()

        # 2) Both with fill_moon_circle in blue
        r1, r2 = ii1.moon[2], ii2.moon[2]
        m1_i, m1_j = ii1.moon[0], ii1.moon[1]
        m2_i, m2_j = ii2.moon[0], ii2.moon[1]
        filled1 = fill_moon_blue_np(arr1, m1_i, m1_j, r1 + 2.0)
        filled2 = fill_moon_blue_np(arr2, m2_i, m2_j, r2 + 2.0)
        crop_f1 = filled1[i_lo:i_hi, j_lo:j_hi]
        crop_f2 = filled2[i_lo:i_hi, j_lo:j_hi]
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
        ax1.imshow(crop_f1)
        ax1.set_title(f"Pair ({i},{j}): first + blue moon fill")
        ax1.axis("off")
        ax2.imshow(crop_f2)
        ax2.set_title(f"second + blue moon fill")
        ax2.axis("off")
        plt.suptitle(f"Pair {ii1.path.name} / {ii2.path.name} — blue moon fill")
        plt.tight_layout()
        plt.show()

        # 3) Aligned and semitransparent (0.5 * first + 0.5 * aligned second)
        blended = 0.5 * crop1.astype(np.float64) + 0.5 * crop2_aligned.astype(np.float64)
        plt.figure(figsize=(8, 8))
        plt.imshow(blended, cmap="gray", vmin=0, vmax=1)
        plt.title(f"Pair ({i},{j}): aligned overlay — {ii1.path.name} / {ii2.path.name}")
        plt.axis("off")
        plt.tight_layout()
        plt.show()